План

* recap seq2seq и трансформеров

* Пишем T5, подгружаем веса с hf, тюним на суммаризацию

### Архитектура T5 (Pre-Norm)
![T5 Architecture](t5_architecture.png)

### Lecture recap

https://github.com/ashaba1in/hse-nlp/blob/2025/week03_transformer/lecture3.pdf

## Пишем свой [T5](https://arxiv.org/abs/1910.10683)

### Attention

In [1]:
import torch
from torch import nn
import math
from einops import rearrange

ModuleNotFoundError: No module named 'einops'

In [38]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


```
token_ids → Embedding → hidden_states
                              ↓
                        Decoder Layer 0 → hidden_states (обновлённый)
                              ↓
                        Decoder Layer 1 → hidden_states (обновлённый)
                              ↓
                            ...
                              ↓
                        Decoder Layer N → hidden_states (финальный)
                              ↓
                         lm_head → logits
```


hidden_state - то что передается между слоями энкодеров и декодеров

In [ ]:
class T5Attention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_kv: int,
        dropout: float,
        is_decoder: bool,
        has_relative_attention_bias: bool,
        num_buckets: int,
        max_distance: int,
    ):
        super().__init__()
        self.is_decoder = is_decoder
        self.has_relative_attention_bias = has_relative_attention_bias

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_kv = d_kv
        self.inner_dim = num_heads * d_kv
        self.dropout = dropout

        self.q = nn.Linear(d_model, self.inner_dim, bias=False)
        self.k = nn.Linear(d_model, self.inner_dim, bias=False)
        self.v = nn.Linear(d_model, self.inner_dim, bias=False)
        self.o = nn.Linear(self.inner_dim, d_model, bias=False)

        if has_relative_attention_bias:
            self.relative_attention_num_buckets = num_buckets
            self.relative_attention_max_distance = max_distance
            self.relative_attention_bias = nn.Embedding(num_buckets, num_heads)

    def _relative_position_bucket(
        self,
        relative_position: torch.Tensor,
        bidirectional: bool,
        num_buckets: int,
        max_distance: int,
    ) -> torch.Tensor:
        relative_buckets = 0
        if bidirectional:
            num_buckets //= 2
            relative_buckets += (relative_position > 0).to(torch.long) * num_buckets
            relative_position = relative_position.abs()
        else:
            relative_position = -torch.min(relative_position, torch.zeros_like(relative_position))

        max_exact = num_buckets // 2
        is_small = relative_position < max_exact
        large_pos = max_exact + (
            torch.log(relative_position.float() / max_exact)
            / math.log(max_distance / max_exact)
            * (num_buckets - max_exact)
        ).to(torch.long)
        large_pos = torch.min(
            large_pos,
            torch.full_like(large_pos, num_buckets - 1),
        )
        relative_buckets += torch.where(is_small, relative_position, large_pos)
        return relative_buckets

    def compute_bias(self, query_len: int, key_len: int, device) -> torch.Tensor:
        """
        Return (1, num_heads, query_len, key_len).
        """
        context_pos = torch.arange(query_len, dtype=torch.long, device=device)[:, None]
        memory_pos = torch.arange(key_len, dtype=torch.long, device=device)[None, :]
        relative_position = memory_pos - context_pos  # (Lq, Lk)

        rp_bucket = self._relative_position_bucket(
            relative_position,
            bidirectional=not self.is_decoder,
            num_buckets=self.relative_attention_num_buckets,
            max_distance=self.relative_attention_max_distance,
        )
        values = self.relative_attention_bias(rp_bucket)  # (Lq - Длина входной последовательности, Lk-Длина ключей, num_heads)
        values = values.permute(2, 0, 1).unsqueeze(0)     # (1, num_heads, Lq, Lk)
        return values

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask=None,
        key_value_states=None,
    ) -> torch.Tensor:
        bsz, q_len, _ = hidden_states.size()                     # hidden_states: (B, Lq = кол_во слов(токенов), d_model (размерность эмебеддов))
        is_cross_attention = key_value_states is not None
        source_states = key_value_states if is_cross_attention else hidden_states
        # source_states: (B, Lk, d_model) where Lk = Lq for self-attn

        q = rearrange(
            self.q(hidden_states),
            "B L (num_heads d_kv) -> B num_heads L d_kv",
            num_heads=self.num_heads,
        )                                                        # q: (B, H, Lq, d_kv-Размерность вектора в одной голове)
        k = rearrange(
            self.k(source_states),
            "B L (num_heads d_kv) -> B num_heads L d_kv",
            num_heads=self.num_heads,
        )                                                        # k: (B, H, Lk, d_kv)
        v = rearrange(
            self.v(source_states),
            "B L (num_heads d_kv) -> B num_heads L d_kv",
            num_heads=self.num_heads,
        )                                                         # v: (B, H, Lk, d_kv)

        scores = torch.matmul(q, rearrange(k, "... L d_kv -> ... d_kv L"))
        # rearrange(k): (B, H, d_kv, Lk)
        # scores: (B, H, Lq, Lk)

        if self.has_relative_attention_bias and not is_cross_attention:
            pos_bias = self.compute_bias(q_len, k.size(-2), device=scores.device)
            # pos_bias: (1, H, Lq, Lk)
            scores = scores + pos_bias                            # scores: (B, H, Lq, Lk) (broadcast по batch)

        if attention_mask is not None:
            # attention_mask обычно: (B, 1, Lq, Lk) или (B, H, Lq, Lk) или (1, 1, Lq, Lk)
            # (additive mask: 0 для разрешённых, -inf для запрещённых)
            scores = scores + attention_mask                      # scores: (B, H, Lq, Lk) (broadcast)

        attn_weights = torch.softmax(scores, dim=-1)              # attn_weights: (B, H, Lq, Lk)
        attn_weights = nn.functional.dropout(
            attn_weights, p=self.dropout, training=self.training
        )                                                        # attn_weights: (B, H, Lq, Lk)

        attn_output = torch.matmul(attn_weights, v)               # attn_output: (B, H, Lq, d_kv)
        attn_output = rearrange(
            attn_output, "B num_heads L d_kv -> B L (num_heads d_kv)"
        )                                                        # attn_output: (B, Lq, H*d_kv) == (B, Lq, inner_dim)
        return self.o(attn_output)  

### LayerNorm

In [40]:
class T5LayerNorm(nn.Module):
    """
    RMSNorm
    """
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        variance = x.to(torch.float32).pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

### Self-Attention и Cross-Attention

In [41]:
class T5LayerSelfAttention(nn.Module):
    def __init__(self, config, is_decoder: bool, has_relative_attention_bias: bool):
        super().__init__()
        self.SelfAttention = T5Attention(
            d_model=config.d_model,
            num_heads=config.num_heads,
            d_kv=config.d_kv,
            dropout=config.dropout_rate,
            is_decoder=is_decoder,
            has_relative_attention_bias=has_relative_attention_bias,
            num_buckets=config.relative_attention_num_buckets,
            max_distance=config.relative_attention_max_distance,
        )
        self.layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, hidden_states: torch.Tensor, attention_mask=None) -> torch.Tensor:
        normed = self.layer_norm(hidden_states)
        attn_output = self.SelfAttention(normed, attention_mask=attention_mask)
        return hidden_states + self.dropout(attn_output)

In [42]:
class T5LayerCrossAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.EncDecAttention = T5Attention(
            d_model=config.d_model,
            num_heads=config.num_heads,
            d_kv=config.d_kv,
            dropout=config.dropout_rate,
            is_decoder=True,
            has_relative_attention_bias=False,
            num_buckets=config.relative_attention_num_buckets,
            max_distance=config.relative_attention_max_distance,
        )
        self.layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        hidden_states: torch.Tensor,
        key_value_states: torch.Tensor,
        attention_mask=None,
    ) -> torch.Tensor:
        normed = self.layer_norm(hidden_states)
        attn_output = self.EncDecAttention(
            normed,
            attention_mask=attention_mask,
            key_value_states=key_value_states,
        )
        return hidden_states + self.dropout(attn_output)

### FFN

In [43]:
class T5DenseReluDense(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.wi = nn.Linear(d_model, d_ff, bias=False)
        self.wo = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.wi(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.wo(x)
        return x
    

class T5LayerFF(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.DenseReluDense = T5DenseReluDense(
            d_model=config.d_model,
            d_ff=config.d_ff,
            dropout=config.dropout_rate,
        )
        self.layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        normed = self.layer_norm(hidden_states)
        ff_output = self.DenseReluDense(normed)
        return hidden_states + self.dropout(ff_output)

### Соединяем всё вместе

In [44]:
class T5Block(nn.Module):
    def __init__(self, config, is_decoder: bool, has_relative_attention_bias: bool):
        super().__init__()
        self.is_decoder = is_decoder
        self.layer = nn.ModuleList()
        self.layer.append(
            T5LayerSelfAttention(
                config,
                is_decoder=is_decoder,
                has_relative_attention_bias=has_relative_attention_bias,
            )
        )
        if is_decoder:
            self.layer.append(T5LayerCrossAttention(config))
        self.layer.append(T5LayerFF(config))

    def forward(
        self,
        hidden_states: torch.Tensor,
        self_attention_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
    ) -> torch.Tensor:
        # self-attention
        hidden_states = self.layer[0](
            hidden_states,
            attention_mask=self_attention_mask,
        )

        # cross-attention if decoder
        if self.is_decoder and encoder_hidden_states is not None:
            hidden_states = self.layer[1](
                hidden_states,
                key_value_states=encoder_hidden_states,
                attention_mask=encoder_attention_mask,
            )

        # FFN
        hidden_states = self.layer[-1](hidden_states)
        return hidden_states

In [45]:
class T5Encoder(nn.Module):
    def __init__(self, config, shared_embedding: nn.Embedding):
        super().__init__()
        self.embed_tokens = shared_embedding
        self.block = nn.ModuleList(
            [
                T5Block(
                    config,
                    is_decoder=False,
                    has_relative_attention_bias=(i == 0),
                )
                for i in range(config.num_layers)
            ]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask=None,
    ) -> torch.Tensor:
        hidden_states = self.embed_tokens(input_ids)  # (B, L, d_model)
        hidden_states = self.dropout(hidden_states)

        # padding mask: (B, 1, 1, L) either 0 or -inf
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :]) * -1e9
        else:
            attn_mask = None

        for block in self.block:
            hidden_states = block(
                hidden_states,
                self_attention_mask=attn_mask,
            )

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return hidden_states  # (B, L, d_model)

In [46]:
class T5Decoder(nn.Module):
    def __init__(self, config, shared_embedding: nn.Embedding):
        super().__init__()
        self.embed_tokens = shared_embedding
        self.block = nn.ModuleList(
            [
                T5Block(
                    config,
                    is_decoder=True,
                    has_relative_attention_bias=(i == 0),
                )
                for i in range(config.num_decoder_layers)
            ]
        )
        self.final_layer_norm = T5LayerNorm(config.d_model, eps=config.layer_norm_epsilon)
        self.dropout = nn.Dropout(config.dropout_rate)

    def _build_self_attention_mask(
        self,
        attention_mask,
        seq_len: int,
        device,
    ) -> torch.Tensor:
        """
        Combining causal and padding masks
        """
        causal_mask = torch.triu(
            torch.ones((seq_len, seq_len), device=device),
            diagonal=1,
        )
        causal_mask = causal_mask[None, None, :, :] * -1e9  # (1, 1, L, L)

        if attention_mask is not None:
            pad_mask = (1.0 - attention_mask[:, None, None, :]) * -1e9  # (B, 1, 1, L)
            mask = causal_mask + pad_mask
        else:
            mask = causal_mask
        return mask

    def _build_encoder_attention_mask(
        self,
        encoder_attention_mask,
    ):
        if encoder_attention_mask is None:
            return None
        return (1.0 - encoder_attention_mask[:, None, None, :]) * -1e9

    def forward(
        self,
        input_ids: torch.Tensor,
        encoder_hidden_states: torch.Tensor,
        attention_mask=None,          # decoder_attention_mask (B, L_dec)
        encoder_attention_mask=None,  # encoder_attention_mask (B, L_enc)
    ) -> torch.Tensor:
        hidden_states = self.embed_tokens(input_ids)
        hidden_states = self.dropout(hidden_states)

        _, tgt_len = input_ids.size()
        device = input_ids.device

        self_attn_mask = self._build_self_attention_mask(
            attention_mask=attention_mask,
            seq_len=tgt_len,
            device=device,
        )
        enc_attn_mask = self._build_encoder_attention_mask(encoder_attention_mask)

        for block in self.block:
            hidden_states = block(
                hidden_states,
                self_attention_mask=self_attn_mask,
                encoder_hidden_states=encoder_hidden_states,
                encoder_attention_mask=enc_attn_mask,
            )

        hidden_states = self.final_layer_norm(hidden_states)
        hidden_states = self.dropout(hidden_states)
        return hidden_states  # (B, L_dec, d_model)

In [47]:
class OurT5ForConditionalGeneration(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model_dim = config.d_model

        self.shared = nn.Embedding(config.vocab_size, config.d_model)

        self.encoder = T5Encoder(config, shared_embedding=self.shared)
        self.decoder = T5Decoder(config, shared_embedding=self.shared)

        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

    def _shift_right(self, labels: torch.Tensor) -> torch.Tensor:
        pad_token_id = self.config.pad_token_id
        decoder_start_token_id = self.config.decoder_start_token_id

        if decoder_start_token_id is None:
            decoder_start_token_id = pad_token_id

        shifted = labels.new_zeros(labels.shape)
        shifted[..., 1:] = labels[..., :-1]
        shifted[..., 0] = decoder_start_token_id

        shifted.masked_fill_(shifted == -100, pad_token_id)
        return shifted

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        labels=None,
        encoder_hidden_states=None,
    ):

        if encoder_hidden_states is None:
            encoder_hidden_states = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        if labels is not None and decoder_input_ids is None:
            decoder_input_ids = self._shift_right(labels)

        decoder_hidden_states = self.decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=decoder_attention_mask,
            encoder_attention_mask=attention_mask,
        )

        logits = self.lm_head(decoder_hidden_states * (self.config.d_model ** -0.5))

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
            )

        return {"loss": loss, "logits": logits}
    
    @torch.no_grad()
    def generate(
        self, 
        input_ids=None,
        attention_mask=None,
        max_length=40,
    ):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        decoder_input_ids_step = torch.tensor([[0]], device=encoder_outputs.device)
        for _ in range(max_length):
            outputs = self(
                encoder_hidden_states=encoder_outputs,
                decoder_input_ids=decoder_input_ids_step,
            )
            logits = outputs["logits"]

            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

            decoder_input_ids_step = torch.cat([decoder_input_ids_step, next_token_id], dim=-1)

            if next_token_id.item() == 1:
                break
        
        return decoder_input_ids_step

In [ ]:
from transformers import AutoModelForSeq2SeqLM

hf_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small") # готовая рабочая модель 
print(hf_model.config) # все гиперпарам модели
our_model = OurT5ForConditionalGeneration(hf_model.config)
our_model.load_state_dict(hf_model.state_dict())

<All keys matched successfully>

Fine-tuning — это дообучение предобученной модели на небольшом датасете для адаптации к конкретной задаче с сохранением ранее выученных знаний.  
При fine-tuning веса модели, полученные на этапе предобучения, используются как начальная точка (а не случайная инициализация), и обновляются с малым learning rate, чтобы сохранить ранее выученные общие знания о языке и при этом специализировать модель под целевую задачу (например, суммаризацию, перевод, классификацию).

### Генерируем настоящей моделью

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-small")

hf_model.cuda()

src_text = "It's a nice day today."
inputs = tokenizer("translate English to German: " + src_text, return_tensors="pt").to("cuda")

generated_ids = hf_model.generate(
    **inputs,
    max_length=40,
)
print("Original text:", src_text)
print("Translation:", tokenizer.decode(generated_ids[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

NameError: name 'hf_model' is not defined

### Теперь проверяем нашу

Помнишь главную идею T5 — "всё есть текст → текст"? Модель обучена на разных задачах одновременно, и чтобы она поняла, какую задачу ты хочешь решить, перед текстом ставится префикс-инструкция:

Суммаризация
"summarize: The Inflation Reduction Act lowers..."
→ краткое содержание


Без префикса модель не знает, что ты от неё хочешь — перевести? суммаризировать? продолжить текст? Префикс — это команда для модели.

Это было зашито при предобучении: Google обучал T5 одновременно на множестве задач, и каждая задача имела свой префикс. Модель научилась: видишь "summarize:" → делай суммаризацию.



In [ ]:
our_model.cuda()

src_text = "It's a nice day today."
inputs = tokenizer("translate English to German: " + src_text, return_tensors="pt").to("cuda")


generated_ids = our_model.generate(
    **inputs,
    max_length=40,
)
print("Original text:", src_text)
print("Translation:", tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Original text: It's a nice day today.
Translation: Heute ist es ein schönes es guter Tag.


## Finetune на summarization

In [52]:
import torch
from datasets import load_dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

In [53]:
billsum = load_dataset("billsum", split="ca_test")
billsum = billsum.train_test_split(test_size=0.2)
billsum["train"][0]

{'text': 'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThis act shall be known, and may be cited, as the Pool Safety Act.\nSEC. 2.\nThe Legislature finds and declares all of the following:\n(a) Swimming pools provide children and their families with a wonderful opportunity for recreation, exercise, and fun. Keeping children safe during this activity is supported by parents and guardians, safety advocates, health providers, insurance companies, and the swimming pool industry.\n(b) According to both the federal Centers for Disease Control and Prevention’s National Center for Injury Prevention and Control and the State Department of Public Health’s EpiCenter data, drowning is the leading cause of death for California children one to four years of age, inclusive.\n(c) Additional children suffer near-drowning incidents and survive, but many of those children suffer irreversible brain injuries, which can lead to lifelong learning deficiencies that impact not on

In [54]:
prefix = "summarize: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [55]:
tokenized_billsum = billsum.map(preprocess_function, batched=True)

 ... (more hidden) ...

 ... (more hidden) ...


In [56]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=our_model)

In [3]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./t5_billsum_ca_test",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    report_to="none",
    num_train_epochs=4,
    save_safetensors=False,
)

trainer = Seq2SeqTrainer(
    model=our_model,
    args=training_args,
    train_dataset=tokenized_billsum["train"],
    eval_dataset=tokenized_billsum["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model("./t5_billsum_ca_test_finetuned")
tokenizer.save_pretrained("./t5_billsum_ca_test_finetuned")

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'save_safetensors'

In [4]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="./t5_billsum_ca_test_finetuned",
    tokenizer="./t5_billsum_ca_test_finetuned",
)

text = "The Inflation Reduction Act lowers prescription drug costs, health care costs, and energy costs. It's the most aggressive action on tackling the climate crisis in American history, which will lift up American workers and create good-paying, union jobs across the country. It'll lower the deficit and ask the ultra-wealthy and corporations to pay their fair share. And no one making under $400,000 per year will pay a penny more in taxes."
print("SOURCE:\n", text)

summary = summarizer("summarize: " + text, max_length=128, min_length=20, do_sample=False)[0]["summary_text"]
print("SUMMARY:\n", summary)

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './t5_billsum_ca_test_finetuned'.